# Harmony360 Governed Market Platform — Recovery + Research Spike v0.2

**Research/paper only. No live orders. No canonical authority.**

This phone-first notebook searches targeted Harmony360 folders, receipts candidate implementations, opens the real GitHub repository, runs real SPY baselines plus experimental FFT/wavelet features, and exports a governed evidence ZIP.

In [ ]:
# Cell 1 — Configuration (instant)
from pathlib import Path
import os, json, time, hashlib, shutil, subprocess, zipfile
from datetime import datetime, timezone
RUN_MODE="QUICK"                 # QUICK | STANDARD | FULL
PAPER_ONLY=True
RESUME=True
FULL_ESTATE_SCAN=False
MAX_FILES={"QUICK":500,"STANDARD":3000,"FULL":15000}[RUN_MODE]
MAX_HASH_BYTES={"QUICK":200_000_000,"STANDARD":1_000_000_000,"FULL":3_000_000_000}[RUN_MODE]
HEARTBEAT_SECONDS=10
DRIVE_ROOT=Path("/content/drive/MyDrive/HAR360_MASTER")
WORK_ROOT=Path("/content/h360_market_spike")
OUTPUT_ROOT=DRIVE_ROOT/"governed_market_platform"/"notebook_runs"
GITHUB_REPOSITORY="https://github.com/rdntmsn/Harmony360.git"
ROOT_HINTS=["canonical","canonical_reconstruction","notebooks","reports","archives","releases","artifacts","snapshots","scripts"]
TARGET_TERMS=("FractalSeerDivision","GuardianDivision","TLN369","MarketState","PortfolioState","calculate_market_uplift","calculate_fractal_resonance")
TARGET_NAMES={"Harmonyv10.ipynb","Harmonyv10_UserJourney.ipynb","Harmony360_v11.ipynb","Harmony360_v12.ipynb","Harmony360_v13.ipynb","harmony360_core_autogen_clean.py"}
WORK_ROOT.mkdir(parents=True,exist_ok=True)
print({"mode":RUN_MODE,"paper_only":PAPER_ONLY,"max_files":MAX_FILES,"full_estate_scan":FULL_ESTATE_SCAN,"next":"Cell 2 — mount Drive"})

In [ ]:
# Cell 2 — Mount Drive (usually under 30 seconds)
try:
    from google.colab import drive
    drive.mount("/content/drive",force_remount=False)
except Exception as exc:
    print("Drive mount skipped:",type(exc).__name__,str(exc)[:160])
if not DRIVE_ROOT.exists(): raise FileNotFoundError(f"Missing: {DRIVE_ROOT}")
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR=OUTPUT_ROOT/RUN_ID; RUN_DIR.mkdir(parents=True,exist_ok=True)
print({"drive_ready":True,"run_id":RUN_ID,"output":str(RUN_DIR),"next":"Cell 3 — helpers"})

In [ ]:
# Cell 3 — Progress and evidence helpers (instant)
def short(p,n=72):
    s=str(p); return s if len(s)<=n else "…"+s[-n+1:]
def status(stage,done=0,total=None,started=None,current=None,warning=None):
    x={"stage":stage,"done":done,"total":total,"elapsed_s":round(time.time()-started,1) if started else 0}
    if current: x["current"]=short(current)
    if warning: x["warning"]=warning
    print(json.dumps(x),flush=True)
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()
def save(name,data):
    p=RUN_DIR/name; p.write_text(json.dumps(data,indent=2,default=str),encoding="utf-8"); return p
print({"helpers_ready":True,"next":"Cell 4 — targeted multi-folder discovery"})

In [ ]:
# Cell 4 — Targeted multi-folder discovery (visible progress)
started=time.time(); roots=[DRIVE_ROOT/x for x in ROOT_HINTS if (DRIVE_ROOT/x).exists()]
if FULL_ESTATE_SCAN and RUN_MODE=="FULL": roots=[DRIVE_ROOT]
candidates=[]; inspected=0; last=started
for root in roots:
    status("discover-root",inspected,MAX_FILES,started,root)
    for base,dirs,files in os.walk(root):
        dirs[:]=[d for d in dirs if not d.startswith(".")]
        for name in files:
            inspected+=1; p=Path(base)/name; low=name.lower()
            if name in TARGET_NAMES or any(x in low for x in ("fractal","qes","market","financial","harmony360_v1")): candidates.append(p)
            if time.time()-last>=HEARTBEAT_SECONDS:
                status("discovering",inspected,MAX_FILES,started,p); last=time.time()
            if inspected>=MAX_FILES: break
        if inspected>=MAX_FILES: break
    if inspected>=MAX_FILES: break
candidates=sorted(set(candidates))
discovery={"roots":[str(x) for x in roots],"inspected":inspected,"candidates":[str(x.relative_to(DRIVE_ROOT)) for x in candidates],"bounded":not(FULL_ESTATE_SCAN and RUN_MODE=="FULL")}
save("discovery.json",discovery)
status("discovery-complete",inspected,inspected,started)
print({"candidate_count":len(candidates),"next":"Cell 5 — hash and classify"})

In [ ]:
# Cell 5 — Hash and classify selected candidates
started=time.time(); receipts=[]; hashed=0; last=started
for i,p in enumerate(candidates,1):
    try:
        size=p.stat().st_size
        if hashed+size>MAX_HASH_BYTES:
            receipts.append({"path":str(p.relative_to(DRIVE_ROOT)),"size_bytes":size,"classification":"DISCOVERED_NOT_HASHED","reason":"MAX_HASH_BYTES"}); continue
        receipts.append({"path":str(p.relative_to(DRIVE_ROOT)),"size_bytes":size,"sha256":sha256_file(p),"classification":"RECOVERY_CANDIDATE" if p.suffix.lower() in {".py",".ipynb",".har360"} else "RESEARCH_OR_REVIEW_EVIDENCE","authority_granted_by_location":False}); hashed+=size
    except Exception as exc: receipts.append({"path":str(p),"classification":"READ_FAILED","error":f"{type(exc).__name__}: {str(exc)[:120]}"})
    if time.time()-last>=HEARTBEAT_SECONDS or i==len(candidates): status("hashing",i,len(candidates),started,p); last=time.time()
save("source_receipts.json",receipts)
print({"receipts":len(receipts),"hashed_mb":round(hashed/1e6,2),"next":"Cell 6 — real GitHub checkout"})

In [ ]:
# Cell 6 — Open the real GitHub repository
checkout=WORK_ROOT/"github_harmony360"; started=time.time()
if checkout.exists() and RESUME: subprocess.run(["git","-C",str(checkout),"pull","--ff-only"],check=False)
else:
    shutil.rmtree(checkout,ignore_errors=True)
    subprocess.run(["git","clone","--depth","1",GITHUB_REPOSITORY,str(checkout)],check=True)
commit=subprocess.check_output(["git","-C",str(checkout),"rev-parse","HEAD"],text=True).strip()
status("github-ready",1,1,started,checkout)
print({"commit":commit,"next":"Cell 7 — implementation scan"})

In [ ]:
# Cell 7 — Locate real target classes and functions
started=time.time(); paths=[p for p in candidates if p.suffix.lower() in {".py",".ipynb",".md",".txt"}]+[p for p in checkout.rglob("*") if p.is_file() and p.suffix.lower() in {".py",".ipynb",".md"}]
matches=[]; warnings=[]; last=started; selected=paths[:MAX_FILES]
for i,p in enumerate(selected,1):
    try:
        text=p.read_text(encoding="utf-8",errors="ignore"); found=[t for t in TARGET_TERMS if t in text]
        if found: matches.append({"path":str(p),"terms":found,"size_bytes":p.stat().st_size})
    except Exception as exc: warnings.append({"path":short(p),"error":type(exc).__name__})
    if time.time()-last>=HEARTBEAT_SECONDS or i==len(selected): status("source-scan",i,len(selected),started,p); last=time.time()
save("implementation_matches.json",matches)
print({"matching_files":len(matches),"warnings":len(warnings),"next":"Cell 8 — dependencies"})

In [ ]:
# Cell 8 — Install research dependencies (pip shows activity)
%pip -q install yfinance PyWavelets
print({"dependencies_ready":True,"next":"Cell 9 — SPY data"})

In [ ]:
# Cell 9 — Download and receipt real SPY daily data
import numpy as np, pandas as pd, yfinance as yf
started=time.time(); status("market-download",0,1,started,"SPY")
raw=yf.download("SPY",start="2005-01-01",auto_adjust=False,progress=True)
if raw.empty: raise RuntimeError("SPY download returned no rows")
if isinstance(raw.columns,pd.MultiIndex): raw.columns=raw.columns.get_level_values(0)
data=raw[["Open","High","Low","Close","Adj Close","Volume"]].dropna().copy(); csv_path=RUN_DIR/"SPY_daily.csv"; data.to_csv(csv_path)
market_receipt={"symbol":"SPY","rows":len(data),"first":str(data.index.min()),"last":str(data.index.max()),"sha256":sha256_file(csv_path),"source":"yfinance/Yahoo; research use only"}
save("market_data_receipt.json",market_receipt); status("market-download",1,1,started,csv_path)
print({**market_receipt,"next":"Cell 10 — baselines"})

In [ ]:
# Cell 10 — Deterministic cost-aware baselines
prices=data["Adj Close"].astype(float); returns=prices.pct_change().fillna(0.0); fee_bps=2.0
signal_ma=(prices.rolling(50).mean()>prices.rolling(200).mean()).astype(float).shift(1).fillna(0.0)
def costed(sig,r): return sig*r-sig.diff().abs().fillna(sig.abs())*fee_bps/10000
def metrics(r):
    r=r.dropna(); eq=(1+r).cumprod(); years=max(len(r)/252,1/252); dd=eq/eq.cummax()-1
    return {"CAGR":float(eq.iloc[-1]**(1/years)-1),"Volatility":float(r.std()*np.sqrt(252)),"Sharpe_0rf":float(r.mean()/r.std()*np.sqrt(252)) if r.std() else 0,"MaxDrawdown":float(dd.min()),"FinalMultiple":float(eq.iloc[-1])}
baseline_results={"buy_hold":metrics(returns),"ma_50_200":metrics(costed(signal_ma,returns))}; save("baseline_results.json",baseline_results)
pd.DataFrame(baseline_results).T.style.format("{:.3f}")

In [ ]:
# Cell 11 — Actual experimental FFT and wavelet features
import pywt
ret=np.log(prices).diff().fillna(0.0)
def fft_ratio(x):
    y=np.asarray(x)-np.mean(x); power=np.abs(np.fft.rfft(y))**2; cutoff=max(2,len(power)//10)
    return float(power[1:cutoff].sum()/power[1:].sum()) if power[1:].sum() else 0.0
def wavelet_ratio(x):
    coeffs=pywt.wavedec(np.asarray(x),"db4",level=3); e=np.array([np.sum(v*v) for v in coeffs])
    return float(e[-1]/e.sum()) if e.sum() else 0.0
features=pd.DataFrame({"fft_low_frequency_ratio":ret.rolling(256).apply(fft_ratio,raw=True),"wavelet_detail_ratio":ret.rolling(256).apply(wavelet_ratio,raw=True)}).dropna(); features.to_csv(RUN_DIR/"experimental_features.csv")
print({"feature_rows":len(features),"classification":"EXPERIMENTAL","production_authority":False,"next":"Cell 12 — walk-forward"})

In [ ]:
# Cell 12 — Out-of-sample ablation
joined=pd.concat([returns.rename("r"),signal_ma.rename("ma"),features],axis=1).dropna(); split=int(len(joined)*.70); train,test=joined.iloc[:split],joined.iloc[split:]
fft_cut=train.fft_low_frequency_ratio.median(); wav_cut=train.wavelet_detail_ratio.median()
s_fft=(test.fft_low_frequency_ratio>=fft_cut).astype(float).shift(1).fillna(0); s_wav=(test.wavelet_detail_ratio<=wav_cut).astype(float).shift(1).fillna(0); s_combo=test.ma*s_fft*s_wav
walk_forward={"test_buy_hold":metrics(test.r),"test_ma":metrics(costed(test.ma,test.r)),"test_fft_filter":metrics(costed(s_fft,test.r)),"test_wavelet_filter":metrics(costed(s_wav,test.r)),"test_combined_ablation":metrics(costed(s_combo,test.r))}
save("walk_forward_results.json",walk_forward)
print({"train_rows":len(train),"test_rows":len(test),"classification":"EXPERIMENTAL_RESULT_NOT_VALIDATED"}); pd.DataFrame(walk_forward).T.style.format("{:.3f}")

In [ ]:
# Cell 13 — Export governed evidence ZIP and summary
summary={"run_id":RUN_ID,"completed_at":datetime.now(timezone.utc).isoformat(),"mode":RUN_MODE,"paper_only":PAPER_ONLY,"github_commit":commit,"roots":discovery["roots"],"files_inspected":discovery["inspected"],"candidate_sources":len(candidates),"source_receipts":len(receipts),"implementation_matches":len(matches),"market_data":market_receipt,"baseline_results":baseline_results,"walk_forward_results":walk_forward,"authority":{"canonical":False,"live_trading":False,"classification":"RESEARCH_IMPLEMENTATION_CANDIDATE"},"next_step":"Human review of recovered implementations and walk-forward evidence"}
save("RUN_SUMMARY.har360.json",summary); zip_path=OUTPUT_ROOT/f"harmony360-market-spike-{RUN_ID}.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(RUN_DIR))
print(json.dumps({"status":"COMPLETE","zip":str(zip_path),"sha256":sha256_file(zip_path),"live_orders_submitted":0,"next":"Review RUN_SUMMARY.har360.json"},indent=2))

## Completion boundary

This notebook performs targeted recovery, source receipt creation, baseline testing, actual FFT/wavelet computation, out-of-sample ablation, and governed export. It does not prove predictive edge or authorize brokerage execution.